#  Lab 1 Assignment : Pipeline Streaming

**Partie 0 : Setup - Imports et Session Spark**

In [ ]:
import os, sys, time, datetime, pathlib, json
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType

# Créer la session Spark
spark = SparkSession.builder \
    .appName("de2-assignment1") \
    .master("local[*]") \
    .getOrCreate()

print("Version de Spark:", spark.version)
print("Interface Spark UI:", spark.sparkContext.uiWebUrl)

# Créer les répertoires de sortie
os.makedirs("outputs/lab1/stream_sink", exist_ok=True)
os.makedirs("outputs/lab1/checkpoint", exist_ok=True)
os.makedirs("proof", exist_ok=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 10:12:19 WARN Utils: Your hostname, Wandaogo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 10:12:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 10:12:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version de Spark: 4.0.1
Interface Spark UI: http://10.255.255.254:4040


**Partie 1 : Définir le Schéma et la Source de Streaming**

In [ ]:
# Définir le schéma des événements
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("event_time", TimestampType(), False),
    StructField("event_type", StringType(), True),
    StructField("value", DoubleType(), True),
])

# Paramètres de streaming
EVENT_TIME_COL = "event_time"
WINDOW_DURATION = "10 minutes"  # Fenêtre de 10 minutes
WATERMARK_DELAY = "5 minutes"   # Retard acceptable pour les données tardives

# Source : lire depuis un répertoire
source_path = "data/streaming_input"
os.makedirs(source_path, exist_ok=True)

# Créer le dataframe de streaming
df_stream = spark.readStream \
    .schema(event_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(source_path)

print("Source de streaming créée")
df_stream.printSchema()

Source de streaming créée
root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- value: double (nullable = true)



**Partie 2 : Agrégation avec Watermark**

In [3]:
# Appliquer le watermark et faire l'agrégation par fenêtre
df_windowed = df_stream \
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY) \
    .groupBy(
        F.window(EVENT_TIME_COL, WINDOW_DURATION),
        "event_type"
    ) \
    .agg(
        F.count("*").alias("nb_events"),
        F.avg("value").alias("avg_value"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value")
    ) \
    .select(
        "window.start",
        "window.end",
        "event_type",
        "nb_events",
        "avg_value",
        "min_value",
        "max_value"
    )

print("Agrégation windowed définie")
df_windowed.printSchema()

Agrégation windowed définie
root
 |-- start: timestamp (nullable = true)
 |-- end: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- nb_events: long (nullable = false)
 |-- avg_value: double (nullable = true)
 |-- min_value: double (nullable = true)
 |-- max_value: double (nullable = true)



**Partie 3 : Écrire le Stream vers Parquet**

In [5]:
# Écrire le stream vers Parquet
query = df_windowed.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "outputs/lab1/stream_sink") \
    .option("checkpointLocation", "outputs/lab1/checkpoint") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Streaming query lancée")
print("Query ID:", query.id)
print("Status:", query.status)

Streaming query lancée
Query ID: a9b8b2a3-bdca-4ef9-8d1f-22fa92188725
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/04/23 10:14:38 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


**Partie 4 : Monitoring et Capture des Métriques**

In [6]:
import time

# Attendre quelques micro-batches
time.sleep(30)

# Afficher le plan d'exécution
print("=== Plan de la requête de streaming ===")
df_windowed.explain("formatted")

# Capturer le plan dans un fichier
with open("proof/plan_streaming.txt", "w") as f:
    f.write("=== PLAN D'EXÉCUTION DU STREAMING ===\n\n")
    f.write(str(df_windowed.explain(mode="formatted")))

# Afficher les métriques du dernier micro-batch
print("\n=== Dernières métriques ===")
progress = query.lastProgress
if progress:
    print(f"Entrées traitées: {progress.get('numInputRows', 0)}")
    print(f"Lignes en sortie: {progress.get('numOutputRows', 0)}")
    print(f"Taille batch: {progress.get('batchDuration', 0)} ms")
    print(f"Latence: {progress.get('processingDelay', 0)} ms")

# Arrêter le stream temporairement pour voir les résultats
query.stop()
print("\nRequête arrêtée")

=== Plan de la requête de streaming ===
== Physical Plan ==
* HashAggregate (12)
+- StateStoreSave (11)
   +- * HashAggregate (10)
      +- StateStoreRestore (9)
         +- * HashAggregate (8)
            +- Exchange (7)
               +- * HashAggregate (6)
                  +- * Project (5)
                     +- * Filter (4)
                        +- EventTimeWatermark (3)
                           +- * Project (2)
                              +- StreamingRelation (1)


(1) StreamingRelation
Output [5]: [event_id#0, user_id#1, event_time#2, event_type#3, value#4]
Arguments: FileSource[data/streaming_input], [event_id#0, user_id#1, event_time#2, event_type#3, value#4]

(2) Project [codegen id : 1]
Output [3]: [event_time#2, event_type#3, value#4]
Input [5]: [event_id#0, user_id#1, event_time#2, event_type#3, value#4]

(3) EventTimeWatermark
Input [3]: [event_time#2, event_type#3, value#4]
Arguments: 1b9bfa99-22f7-4224-a30d-481eb0bd1a39, event_time#2: timestamp, 5 minutes

(4) Fi

26/04/23 10:15:42 WARN DAGScheduler: Failed to cancel job group 610f6ce3-2a5e-43dd-aaf2-6d929f1eee47. Cannot find active jobs for it.
26/04/23 10:15:42 WARN DAGScheduler: Failed to cancel job group 610f6ce3-2a5e-43dd-aaf2-6d929f1eee47. Cannot find active jobs for it.


**Partie 5 : Optimisation et Ré-mesure**

In [8]:
# Configuration optimisée (exemple : fenêtre plus courte)
WINDOW_DURATION_OPT = "5 minutes"
WATERMARK_DELAY_OPT = "2 minutes"

df_windowed_opt = df_stream \
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY_OPT) \
    .groupBy(
        F.window(EVENT_TIME_COL, WINDOW_DURATION_OPT),
        "event_type"
    ) \
    .agg(
        F.count("*").alias("nb_events"),
        F.avg("value").alias("avg_value")
    )

# Lancer la requête optimisée
query_opt = df_windowed_opt.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "outputs/lab1/stream_sink_optimized") \
    .option("checkpointLocation", "outputs/lab1/checkpoint_optimized") \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(30)

# Capturer les nouvelles métriques
progress_opt = query_opt.lastProgress
query_opt.stop()

print("\n=== Métriques optimisées ===")
if progress_opt:
    print(f"Entrées traitées: {progress_opt.get('numInputRows', 0)}")
    print(f"Latence: {progress_opt.get('processingDelay', 0)} ms")

26/04/23 10:16:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.



=== Métriques optimisées ===
Entrées traitées: 0
Latence: 0 ms


26/04/23 10:17:16 WARN DAGScheduler: Failed to cancel job group 1e0b1587-6d60-43da-a542-b6da0c73d915. Cannot find active jobs for it.
26/04/23 10:17:16 WARN DAGScheduler: Failed to cancel job group 1e0b1587-6d60-43da-a542-b6da0c73d915. Cannot find active jobs for it.


**Partie 6 : Remplir le Log des Métriques**


In [9]:
import pandas as pd

# Créer le log des métriques
metrics_data = {
    "Configuration": ["Baseline", "Optimized"],
    "Window_Duration": [WINDOW_DURATION, WINDOW_DURATION_OPT],
    "Watermark_Delay": [WATERMARK_DELAY, WATERMARK_DELAY_OPT],
    "Trigger_Interval_ms": [5000, 3000],
    "Input_Rows": [
        progress.get('numInputRows', 0) if progress else 0,
        progress_opt.get('numInputRows', 0) if progress_opt else 0
    ],
    "Processing_Delay_ms": [
        progress.get('processingDelay', 0) if progress else 0,
        progress_opt.get('processingDelay', 0) if progress_opt else 0
    ]
}

metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv("lab1_metrics_log.csv", index=False)

print("Métriques sauvegardées dans lab1_metrics_log.csv")
print(metrics_df)

Métriques sauvegardées dans lab1_metrics_log.csv
  Configuration Window_Duration Watermark_Delay  Trigger_Interval_ms  \
0      Baseline      10 minutes       5 minutes                 5000   
1     Optimized       5 minutes       2 minutes                 3000   

   Input_Rows  Processing_Delay_ms  
0           0                    0  
1           0                    0  


**Partie 7 : Cleanup**


In [10]:
# Arrêter tous les streams
query.stop()
query_opt.stop()

# Arrêter la session Spark
spark.stop()

print("Pipeline de streaming terminé.")
print("Répertoires générés:")
print("  - outputs/lab1/stream_sink/")
print("  - outputs/lab1/checkpoint/")
print("  - proof/plan_streaming.txt")
print("  - lab1_metrics_log.csv")

Pipeline de streaming terminé.
Répertoires générés:
  - outputs/lab1/stream_sink/
  - outputs/lab1/checkpoint/
  - proof/plan_streaming.txt
  - lab1_metrics_log.csv
